# M5 Forecasting — Accuracy

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import jax.numpy as jnp
import pandas as pd
import matplotlib.pyplot as plt

from wunkui.models.m5.m5_ssp_optim import fit_one_series_opt, predict_one_series_opt

DATA_DIR = Path("../resource/m5-forecasting-accuracy")
HORIZON = 28  # 28-day forecast horizon
SEED = 2026

N_DEMO = 2
# "top"    — highest total sales volume
# "random" — uniform random sample (reproducible via SEED)
SAMPLE_MODE = "random"

rng = np.random.default_rng(SEED)

## 1. Load Data

In [ ]:
import xarray as xr
ds = xr.open_dataset(DATA_DIR / "m5_ds.nc")

## 2. Build sales matrix (n_series, n_steps)

In [ ]:
train_mask = ds["split"].values == "train"
sales_matrix = ds["sales"].values[:, train_mask]  # (n_series, n_train_steps)
series_ids = np.array([f"{sid}_validation" for sid in ds["series_id"].values])

print(f"sales_matrix: {sales_matrix.shape}  (n_series={sales_matrix.shape[0]}, n_steps={sales_matrix.shape[1]})")
print(f"series_ids:   {series_ids[:3]} …")

## 3. Build day-of-week design matrix Z

In [ ]:
def _dow_dummies(dow: np.ndarray) -> np.ndarray:
    """Day-of-week dummies with Monday dropped as the reference category."""
    dummies = pd.get_dummies(dow, dtype=np.float32).reindex(columns=range(7), fill_value=0.0)
    return dummies.iloc[:, 1:].values  # drop Mon → (n, 6)

dow = ds["day_of_week"].values  # (n_dates,) int8, Mon=0 … Sun=6
val_mask = ds["split"].values == "validation"

Z_shared        = _dow_dummies(dow[train_mask])  # (1913, 6)
Z_future_shared = _dow_dummies(dow[val_mask])    # (28, 6)

Z_shared = jnp.asarray(Z_shared)
Z_future_shared = jnp.asarray(Z_future_shared)

print(f"Z_shared:        {Z_shared.shape}")
print(f"Z_future_shared: {Z_future_shared.shape}")

In [ ]:
total_sales = sales_matrix.sum(axis=1)

if SAMPLE_MODE == "random":
    demo_idx = rng.choice(len(series_ids), size=N_DEMO, replace=False)
    label = f"random sample (seed={SEED})"
else:
    demo_idx = np.argsort(total_sales)[::-1][:N_DEMO]
    label = "top by volume"

for rank, idx in enumerate(demo_idx):
    print(f"  #{rank + 1}  {series_ids[idx]}  total={total_sales[idx]:,.0f}  [{label}]")

In [ ]:
# Fit and forecast each of the selected series
demo_results = {}
for rank, idx in enumerate(demo_idx):
    sid = series_ids[idx]
    print(f"\n[{rank + 1}/{N_DEMO}] Fitting {sid} …")
    fit_result = fit_one_series_opt(
        sales_matrix[idx], n_iter=1500, lr=3e-2, Z=Z_shared,
    )
    forecast = predict_one_series_opt(fit_result, Z_future=Z_future_shared)
    demo_results[sid] = {"idx": idx, "fit": fit_result, "forecast": forecast}

In [ ]:
# Convergence diagnostics — loss curves for each fitted series
fig, axes = plt.subplots(1, len(demo_results), figsize=(6 * len(demo_results), 3), squeeze=False)
axes = axes[0]

for ax, (sid, res) in zip(axes, demo_results.items()):
    losses = res["fit"]["losses"]
    n_run = res["fit"]["n_iter_run"]
    ax.plot(losses, linewidth=1)
    ax.axvline(n_run - 1, color="tomato", linestyle="--", linewidth=1, label=f"stopped @ {n_run}")
    ax.set_title(sid, fontsize=8, loc="left")
    ax.set_xlabel("Step")
    ax.set_ylabel("Neg log-posterior")
    ax.legend(fontsize=8)

fig.suptitle("Optimiser convergence", y=1.02)
plt.tight_layout()

In [ ]:
# Plot last 90 days + 28-day forecast for each
tail = 90
fig, axes = plt.subplots(N_DEMO, 1, figsize=(12, 3 * N_DEMO), sharex=True, squeeze=False)
axes = axes[:, 0]
for ax, (sid, res) in zip(axes, demo_results.items()):
    actual = sales_matrix[res["idx"]]
    ax.plot(range(tail), actual[-tail:], label="Actual", alpha=0.7)
    ax.plot(range(tail, tail + HORIZON), res["forecast"], label="Forecast", linestyle="--", color="tomato")
    ax.axvline(tail, color="grey", linestyle=":", alpha=0.5)
    ax.set_title(sid, fontsize=9, loc="left")
    ax.set_ylabel("Units")
    ax.legend(fontsize=8)
axes[-1].set_xlabel("Day")
fig.suptitle(f"{N_DEMO} series ({label}) — SSP forecast", y=1.01)
plt.tight_layout()

## 4. Build Submission

The submission needs both **validation** (items ending `_validation`) and **evaluation** (`_evaluation`) rows — 60,980 rows total.

In [ ]:
def make_submission(
    predictions: pd.DataFrame,
    sample_submission: pd.DataFrame,
    output_path: Path | str = "submission.csv",
) -> pd.DataFrame:
    """Build a competition-ready submission CSV.

    Parameters
    ----------
    predictions : pd.DataFrame
        Forecast from ``predict()`` with ``id`` (validation ids) and F1–F28.
    sample_submission : pd.DataFrame
        Official sample_submission template to ensure correct row order.
    output_path : Path | str
        Where to write the CSV.

    Returns
    -------
    pd.DataFrame
        The final submission dataframe.
    """
    output_path = Path(output_path)
    f_cols = [f"F{i}" for i in range(1, HORIZON + 1)]

    # Validation rows: use predictions directly
    val_preds = predictions.copy()
    val_preds["id"] = val_preds["id"].str.replace("_validation", "_validation", regex=False)

    # Evaluation rows: duplicate predictions (same forecast for both horizons as placeholder)
    eval_preds = predictions.copy()
    eval_preds["id"] = eval_preds["id"].str.replace("_validation", "_evaluation", regex=False)

    full = pd.concat([val_preds, eval_preds], ignore_index=True)

    # Align to official submission order
    sub = sample_submission[["id"]].merge(full, on="id", how="left")

    assert sub.shape[0] == sample_submission.shape[0], (
        f"Row count mismatch: got {sub.shape[0]}, expected {sample_submission.shape[0]}"
    )
    assert not sub[f_cols].isna().any().any(), "Missing forecasts detected"

    sub.to_csv(output_path, index=False)
    print(f"Submission written to {output_path}  ({sub.shape[0]} rows)")
    return sub

In [ ]:
# Generate and save submission
sub = make_submission(predictions, submission, output_path="submission.csv")
sub.head()